# DiffusionGemma-Jev (`djev`) on Google Colab Pro (`A100` / `L4`)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/taeold/djev-run/blob/main/colab.ipynb)

> **Colab Pro `A100` / `L4` Required by Default:** This notebook requests an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** (`gpuClass: premium`, `machine_shape: hm`) so all `17.53 GiB` of weights stay 100% in GPU VRAM (`~45 ms` on A100 HBM2e, `~65 ms` on L4). If Colab connects you to a default `T4`, click **`Runtime -> Change runtime type -> Hardware accelerator -> A100 GPU or L4 GPU`** (or top-right dropdown arrow next to `T4 -> Change runtime type`).

Run **DiffusionGemma-Jev** (`nvidia/diffusiongemma-26B-A4B-it-NVFP4`, `26B` total parameters, `4B` active per token across `128` experts) directly inside Google Colab Pro on an **NVIDIA A100 (`40 GB` / `80 GB`)** or **NVIDIA L4 (`24 GB`)** GPU using standard `vLLM` (`POST /tokenize` + `POST /v1/chat/completions` with `extra_body.vllm_xargs`), and play the built-in 1-step diffusion games (`/tetris`, `/dino`, and `/snake`) live inside the notebook.

| Colab Runtime | GPU / Compute Capability | Usable VRAM | `nvidia/diffusiongemma-26B-A4B-it-NVFP4` (`17.53 GiB`) | Active vLLM Kernel Path |
| :--- | :--- | :--- | :--- | :--- |
| **Colab Pro (`A100`)** | NVIDIA A100 (`SM 8.0`) | `40.0 GiB` / `80.0 GiB` | **Recommended (`~45 ms/step`, 100% VRAM)** (`KV_CACHE_GB=8`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Pro (`L4`)** | NVIDIA L4 (`SM 8.9`) | `22.5 GiB` (`24 GB`) | **Supported (`~65 ms/step`, 100% VRAM)** (`KV_CACHE_GB=1.5`) | **Marlin `W4A16` NVFP4 MoE** (`MarlinExperts`) + `BF16` Attention |
| **Colab Free (`T4`)** | NVIDIA T4 (`SM 7.5`) | `15.0 GiB` (`16 GB`) | **Blocked by default** (`ALLOW_SLOW_T4_OFFLOAD = False`; requires `5 GB` PCIe CPU offload at `~400-600 ms/step`) | Switch to `A100` or `L4` in `Runtime -> Change runtime type` |

In [1]:
import os
import subprocess

ALLOW_SLOW_T4_OFFLOAD = False

try:
    smi_out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,memory.free,compute_cap", "--format=csv,noheader,nounits"],
        text=True,
    ).strip().splitlines()[0]
    gpu_name, total_mib, free_mib, compute_cap = [x.strip() for x in smi_out.split(",")]
    vram_gib = float(total_mib) / 1024.0
    free_gib = float(free_mib) / 1024.0
    sm_major, sm_minor = [int(x) for x in compute_cap.split(".")]
except Exception as e:
    raise RuntimeError(
        "No GPU detected via nvidia-smi. In Colab, click Runtime -> Change runtime type -> "
        "Hardware accelerator -> select A100 GPU or L4 GPU."
    ) from e

if vram_gib < 22.0 and not ALLOW_SLOW_T4_OFFLOAD:
    raise RuntimeError(
        "Detected NVIDIA T4 (15 GB). T4 requires 5 GB PCIe CPU offload (~400-600 ms/step). "
        "Please switch to an A100 or L4 GPU via: Runtime -> Change runtime type "
        "-> Hardware accelerator -> A100 GPU or L4 GPU (or set ALLOW_SLOW_T4_OFFLOAD = True in this cell)."
    )

if vram_gib < 22.0:
    KV_CACHE_GB = 0.5
    CPU_OFFLOAD_GB = 5.0
    GPU_UTIL = 0.85
    DTYPE = "float16"
elif vram_gib < 30.0:
    KV_CACHE_GB = 1.5
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.90
    DTYPE = "auto"
elif vram_gib < 60.0:
    KV_CACHE_GB = 8.0
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.85
    DTYPE = "auto"
else:
    KV_CACHE_GB = 8.0
    CPU_OFFLOAD_GB = 0.0
    GPU_UTIL = 0.40
    DTYPE = "auto"

os.environ["KV_CACHE_GB"] = str(KV_CACHE_GB)
os.environ["CPU_OFFLOAD_GB"] = str(CPU_OFFLOAD_GB)
os.environ["GPU_UTIL"] = str(GPU_UTIL)
os.environ["DTYPE"] = DTYPE
os.environ["CANVAS"] = "256"
os.environ["DISABLE_MM"] = "1"

print(f"GPU Device         : {gpu_name} (SM {sm_major}.{sm_minor})")
print(f"Total / Free VRAM  : {vram_gib:.2f} GiB total / {free_gib:.2f} GiB free (0.00 GiB used by notebook kernel)")
print(f"Checkpoint Weights : 17.53 GiB (nvidia/diffusiongemma-26B-A4B-it-NVFP4)")
print(f"MoE Kernel Backend : vLLM Marlin W4A16 FP4 (dtype={DTYPE}, gpu_util={GPU_UTIL}, cpu_offload_gb={CPU_OFFLOAD_GB})")
print(f"Configured KV Cache: {KV_CACHE_GB} GiB (CANVAS=256, DISABLE_MM=1)")

GPU Device         : NVIDIA A100-SXM4-40GB (SM 8.0)
Total / Free VRAM  : 39.56 GiB total / 39.56 GiB free (0.00 GiB used by notebook kernel)
Checkpoint Weights : 17.53 GiB (nvidia/diffusiongemma-26B-A4B-it-NVFP4)
MoE Kernel Backend : vLLM Marlin W4A16 FP4 (dtype=auto, gpu_util=0.85, cpu_offload_gb=0.0)
Configured KV Cache: 8.0 GiB (CANVAS=256, DISABLE_MM=1)


## Step 1: Install `djev-run` & Start Standalone `vLLM` Server

The cell below installs `vllm-openai:nightly`, downloads `server.py`, and spins up `vllm serve`. If you deployed to Cloud Run, simply enter your `CLOUD_RUN_URL` instead!

In [ ]:
import os
import subprocess
import urllib.request
import json
import time

# Paste your Cloud Run URL here (e.g. 'https://djev-dgemma-...run.app')
# If left blank, it will natively install vllm and serve from Colab's GPU
CLOUD_RUN_URL = ""  

if CLOUD_RUN_URL:
    DJEV_BASE_URL = CLOUD_RUN_URL
    print(f"Connected to Cloud Run: {DJEV_BASE_URL}")
else:
    DJEV_BASE_URL = "http://127.0.0.1:8080"
    try:
        import vllm
    except ImportError:
        print("Installing nightly vLLM... (takes ~2 minutes)")
        subprocess.run(["pip", "install", "-U", "vllm", "--pre", "--extra-index-url", "https://wheels.vllm.ai/nightly", "fastapi", "uvicorn"]).check_returncode()
    
    if not os.path.exists("server.py"):
        for f in ["server.py", "snake.html", "dino.html", "tetris.html"]:
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/taeold/djev-run/main/{f}", f)
    
    print("Starting vLLM server in background... (Wait ~3m for PyTorch kernels)")
    os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"
    proc = subprocess.Popen(
        "python3 -m vllm.entrypoints.openai.api_server --model nvidia/diffusiongemma-26B-A4B-it-NVFP4 --middleware server.SystemOneMiddleware --port 8080 --trust-remote-code --enforce-eager --language-model-only --attention-backend TRITON_ATTN --kv-cache-memory 2G --max-num-seqs 32 --max-model-len 4096".split(),
        stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
    )
    while True:
        try:
            if json.loads(urllib.request.urlopen(f"{DJEV_BASE_URL}/health", timeout=1).read().decode())["name"] == "vllm":
                print("\n[djev] Server is ready on Colab!")
                break
        except Exception:
            print(".", end="", flush=True)
            time.sleep(5)


## Step 2: 1-Step Diffusion Canvas Read via Standard `vLLM` (`POST /tokenize` + `POST /v1/chat/completions`)

Tokenize an output template (`POST /tokenize`), pin every scaffold and label token (`diffusion_pinned`), leave the answer slots unpinned (`department`, `urgency`, `refund_requested`), and read all three slot probability distributions simultaneously in **1 forward pass** (`diffusion_max_steps: 1`, `diffusion_read_only: True`).

In [ ]:
import json
import math
import time
import urllib.request

SCAFFOLD = [100, 45518, 107, 101]  # <thought>\n</thought>
template_str = "department: a\nurgency: 1\nrefund_requested: yes"

# 1. Tokenize template via POST /tokenize
tok_req = urllib.request.Request(
    f"{DJEV_BASE_URL}/tokenize",
    data=json.dumps({"prompt": template_str, "add_special_tokens": False}).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer mock"},
)
base_ids = json.loads(urllib.request.urlopen(tok_req, timeout=10).read().decode())["tokens"]
full_template = SCAFFOLD + base_ids

unpinned = [i - 1 for i, t in enumerate(full_template) if i >= 4 and t == 107] + [len(full_template) - 1]
pinned = [i for i in range(len(full_template)) if i not in unpinned]
# Canvas padded to 128 elements securely
seed_canvas = [full_template[i] if i in pinned else (256000 + i * 131) for i in range(len(full_template))]
seed_canvas = seed_canvas + [256000] * (128 - len(seed_canvas))

# 2. Run 1-step read-only diffusion forward pass via POST /v1/chat/completions
chat_payload = {
    "model": "djev-dgemma",
    "messages": [
        {
            "role": "system",
            "content": (
                "Answer each question about the ticket state with its single label.\n"
                "department: a = billing, b = technical, c = sales\n"
                "urgency: 1 = low, 2 = minor, 3 = locked production access, 4 = complete outage\n"
                "refund_requested: yes or no"
            ),
        },
        {
            "role": "user",
            "content": json.dumps({
                "ticket_id": "TCK-9042",
                "text": "I was double-charged $149.00 on invoice INV-2026-8841. Please refund the duplicate charge.",
            }),
        },
    ],
    "max_tokens": len(full_template) + 1,
    "logprobs": True,
    "top_logprobs": 16,
    "vllm_xargs": {
        "diffusion_seed_canvas": seed_canvas,
        "diffusion_pinned": pinned,
        "diffusion_max_steps": 1,
        "diffusion_read_only": True,
    }
}

t0 = time.time()
req = urllib.request.Request(
    f"{DJEV_BASE_URL}/v1/chat/completions",
    data=json.dumps(chat_payload).encode(),
    headers={"Content-Type": "application/json", "Authorization": "Bearer mock"},
)
resp = json.loads(urllib.request.urlopen(req, timeout=10).read().decode())
rtt_ms = round((time.time() - t0) * 1000, 1)
print(f"Completed in {rtt_ms} ms.\nLogprobs output:\n")
print(json.dumps(resp["choices"][0]["logprobs"]["content"], indent=2))


## Step 3: Play `/tetris`, `/dino`, and `/snake` Live Inside Colab

Set `DEMO = "/tetris"`, `"/dino"`, or `"/snake"` and run the cell below to embed the live game UI served from port `8080` on your Colab GPU.

In [ ]:
# Choose a built-in game: "/tetris", "/dino", or "/snake"
DEMO = "/tetris"

try:
    from google.colab import output
    from IPython.display import display, HTML
    print(f"Embedding {DEMO} ...")
    if CLOUD_RUN_URL:
        display(HTML(f'<iframe src="{DJEV_BASE_URL}{DEMO}" width="100%" height="660" frameborder="0"></iframe>'))
    else:
        output.serve_kernel_port_as_iframe(8080, path=DEMO, height=660)
except ImportError:
    print("Available demo routes:")
    for r in ("/tetris", "/dino", "/snake"):
        print(f"  -> {DJEV_BASE_URL}{r}")
